# Class and Static Methods — Advanced Tutorial Problems II

This notebook is a **second, independent problem set** on the same topic.

The style deliberately follows a tutorial progression:

- introduce one idea,
- run a small experiment,
- inspect what Python is doing,
- make a prediction,
- solve one part of a larger problem,
- refine the design,
- then combine the pieces.

The main topic is still the distinction between:

- **instance methods**
- **class methods**
- **static methods**

but the problems go further into inheritance, binding, factories, class configuration, registries, validation, and testable design.


## How to use this notebook

For each problem, try to stop at the **Try it yourself** cell before reading the solution.

Most larger problems are intentionally split into several logical steps.  
That matters because method design is easier to understand when we ask one question at a time:

1. **What information does this operation need?**
2. **Where does that information live?**
3. **Should Python bind an instance, a class, or nothing?**

We will repeatedly use that reasoning instead of memorizing decorators.


## A small vocabulary reminder

If a function is defined in a class body, the way it behaves later depends on the object stored in the class namespace.

A normal function can become a **bound instance method** when retrieved through an instance.

A `classmethod` binds the **class**.

A `staticmethod` performs **no automatic binding**.

We will begin by observing that behavior directly.


# Tutorial Problem 1 — Follow the Binding

We will start with a deliberately small class.

The goal is not merely to call the methods.  
The goal is to understand what Python gives us when the same attribute is accessed in different ways.


In [1]:
class BindingDemo:
    def instance_method(self):
        return f"instance={self!r}"

    @classmethod
    def class_method(cls):
        return f"class={cls.__name__}"

    @staticmethod
    def static_method():
        return "no automatic first argument"


demo = BindingDemo()


## Step 1 — Access the attributes without calling them

Before adding parentheses, inspect the attributes themselves.

Predict which expressions will display as a plain function and which will display as a bound method.


In [2]:
print("From the class:")
print("  BindingDemo.instance_method ->", BindingDemo.instance_method)
print("  BindingDemo.class_method    ->", BindingDemo.class_method)
print("  BindingDemo.static_method   ->", BindingDemo.static_method)

print("\nFrom the instance:")
print("  demo.instance_method        ->", demo.instance_method)
print("  demo.class_method           ->", demo.class_method)
print("  demo.static_method          ->", demo.static_method)


From the class:
  BindingDemo.instance_method -> <function BindingDemo.instance_method at 0x0000024AA7C1D440>
  BindingDemo.class_method    -> <bound method BindingDemo.class_method of <class '__main__.BindingDemo'>>
  BindingDemo.static_method   -> <function BindingDemo.static_method at 0x0000024AA7C2D120>

From the instance:
  demo.instance_method        -> <bound method BindingDemo.instance_method of <__main__.BindingDemo object at 0x0000024AA7B8EBA0>>
  demo.class_method           -> <bound method BindingDemo.class_method of <class '__main__.BindingDemo'>>
  demo.static_method          -> <function BindingDemo.static_method at 0x0000024AA7C2D120>


## Observation

A normal method is interesting because its behavior changes depending on the lookup:

- `BindingDemo.instance_method` is retrieved from the class,
- `demo.instance_method` is retrieved through an instance.

The second lookup produces a method bound to `demo`.

A class method behaves differently.  
Whether it is reached through the class or through an instance, it is bound to a **class object**.

A static method remains function-like.


## Step 2 — Inspect `__self__`

Bound method objects expose the object they are bound to through `__self__`.

Let's make the binding visible.


In [3]:
print("demo.instance_method.__self__ is demo:",
      demo.instance_method.__self__ is demo)

print("BindingDemo.class_method.__self__ is BindingDemo:",
      BindingDemo.class_method.__self__ is BindingDemo)

print("demo.class_method.__self__ is BindingDemo:",
      demo.class_method.__self__ is BindingDemo)


demo.instance_method.__self__ is demo: True
BindingDemo.class_method.__self__ is BindingDemo: True
demo.class_method.__self__ is BindingDemo: True


A static method is different: the result of attribute lookup is not a bound method, so there is no meaningful `__self__` to inspect.

We can verify that with `inspect`.


In [4]:
import inspect

print("instance from class:",
      inspect.isfunction(BindingDemo.instance_method),
      inspect.ismethod(BindingDemo.instance_method))

print("instance from object:",
      inspect.isfunction(demo.instance_method),
      inspect.ismethod(demo.instance_method))

print("class method:",
      inspect.isfunction(BindingDemo.class_method),
      inspect.ismethod(BindingDemo.class_method))

print("static method:",
      inspect.isfunction(BindingDemo.static_method),
      inspect.ismethod(BindingDemo.static_method))


instance from class: True False
instance from object: False True
class method: False True
static method: True False


## Step 3 — Look inside the class dictionary

Attribute access can hide some of the mechanism.

The raw class dictionary lets us see what was stored during class creation.


In [5]:
for name in ("instance_method", "class_method", "static_method"):
    raw = BindingDemo.__dict__[name]
    print(f"{name:16} -> {raw!r}")
    print(f"{'':16}    raw type: {type(raw).__name__}")


instance_method  -> <function BindingDemo.instance_method at 0x0000024AA7C1D440>
                    raw type: function
class_method     -> <classmethod(<function BindingDemo.class_method at 0x0000024AA7C2D080>)>
                    raw type: classmethod
static_method    -> <staticmethod(<function BindingDemo.static_method at 0x0000024AA7C2D120>)>
                    raw type: staticmethod


## What did the decorators actually do?

Notice the difference:

- the ordinary method is stored as a **function**,
- `@classmethod` stores a **classmethod descriptor**,
- `@staticmethod` stores a **staticmethod descriptor**.

So these decorators are not comments or labels.  
They change the object stored in the class namespace.


## Step 4 — Manually ask the descriptors to bind

Python's descriptor protocol uses `__get__`.

We do not normally call it ourselves, but doing so once makes the mechanism much easier to understand.


In [6]:
raw_instance = BindingDemo.__dict__["instance_method"]
raw_class = BindingDemo.__dict__["class_method"]
raw_static = BindingDemo.__dict__["static_method"]

manual_instance = raw_instance.__get__(demo, BindingDemo)
manual_class = raw_class.__get__(demo, BindingDemo)
manual_static = raw_static.__get__(demo, BindingDemo)

print(manual_instance)
print(manual_class)
print(manual_static)

assert manual_instance.__self__ is demo
assert manual_class.__self__ is BindingDemo
assert inspect.isfunction(manual_static)


<bound method BindingDemo.instance_method of <__main__.BindingDemo object at 0x0000024AA7B8EBA0>>
<bound method BindingDemo.class_method of <class '__main__.BindingDemo'>>
<function BindingDemo.static_method at 0x0000024AA7C2D120>


## Problem 1 conclusion

A useful mental model is:

- normal function in class + instance lookup → bind the **instance**
- `classmethod` + lookup → bind the **class**
- `staticmethod` + lookup → return the underlying function without binding

This foundation will explain the design choices in every later problem.


# Tutorial Problem 2 — Alternative Constructors That Respect Inheritance

Suppose we are modeling two-dimensional vectors.

The normal constructor accepts numeric components:

```python
Vector(x, y)
```

But our application also receives values such as:

```text
"10.5, -3.25"
```

We want a convenient constructor:

```python
Vector.from_text("10.5, -3.25")
```

The important question is:

> Should `from_text` be an instance method, class method, or static method?


## Step 1 — Build the normal constructor

There is no decorator question yet.  
We first make the object itself.


In [7]:
class Vector:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __repr__(self):
        return f"{type(self).__name__}(x={self.x}, y={self.y})"


v = Vector(3, 4)
v


Vector(x=3.0, y=4.0)

## Step 2 — A tempting static method

At first, parsing the string looks independent of any instance.

So we might write:


In [8]:
class VectorStaticAttempt:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    @staticmethod
    def from_text(text):
        left, right = text.split(",", maxsplit=1)
        return VectorStaticAttempt(left.strip(), right.strip())


VectorStaticAttempt.from_text("10.5, -3.25").__dict__


{'x': 10.5, 'y': -3.25}

This works for the base class.

But let's introduce inheritance before deciding that the design is correct.


In [9]:
class ColoredVectorStaticAttempt(VectorStaticAttempt):
    pass


result = ColoredVectorStaticAttempt.from_text("1, 2")

print(type(result).__name__)


VectorStaticAttempt


## The problem

We called the constructor through `ColoredVectorStaticAttempt`, but the implementation hard-coded `VectorStaticAttempt(...)`.

The method has no access to the runtime class that was used for the call.

This is exactly the kind of situation where `classmethod` becomes useful.


## Step 3 — Replace the hard-coded class with `cls`

Now the alternative constructor can create whichever class is actually receiving the call.


In [10]:
class Vector:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    @classmethod
    def from_text(cls, text):
        left, right = text.split(",", maxsplit=1)
        return cls(left.strip(), right.strip())

    def __repr__(self):
        return f"{type(self).__name__}(x={self.x}, y={self.y})"


class ColoredVector(Vector):
    pass


base = Vector.from_text("10, 20")
child = ColoredVector.from_text("30, 40")

print(base)
print(child)

assert type(base) is Vector
assert type(child) is ColoredVector


Vector(x=10.0, y=20.0)
ColoredVector(x=30.0, y=40.0)


## Step 4 — Add another constructor without duplicating construction logic

Now suppose data may arrive as a mapping:

```python
{"x": 2, "y": 7}
```

Instead of repeating construction rules, let one class method delegate to another.


In [11]:
class Vector:
    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    @classmethod
    def from_text(cls, text):
        left, right = text.split(",", maxsplit=1)
        return cls(left.strip(), right.strip())

    @classmethod
    def from_mapping(cls, mapping):
        return cls(mapping["x"], mapping["y"])

    @classmethod
    def from_pair(cls, pair):
        x, y = pair
        return cls(x, y)

    def __repr__(self):
        return f"{type(self).__name__}(x={self.x}, y={self.y})"


## Step 5 — Extend construction in a subclass

A subclass may need extra information.

We will create `NamedVector`.  
The parsing method will still use `cls`, but the subclass will override one constructor to supply its additional field.


In [12]:
class NamedVector(Vector):
    def __init__(self, x, y, name="anonymous"):
        super().__init__(x, y)
        self.name = name

    @classmethod
    def from_named_text(cls, text):
        name, coordinates = text.split(":", maxsplit=1)
        x, y = coordinates.split(",", maxsplit=1)
        return cls(x.strip(), y.strip(), name=name.strip())

    def __repr__(self):
        return (
            f"{type(self).__name__}("
            f"x={self.x}, y={self.y}, name={self.name!r})"
        )


nv = NamedVector.from_named_text("velocity: 7, -2")
print(nv)

assert isinstance(nv, NamedVector)
assert nv.name == "velocity"


NamedVector(x=7.0, y=-2.0, name='velocity')


## Problem 2 takeaway

Alternative constructors are usually class methods because they answer two questions at once:

1. how do we transform the input?
2. which class should be instantiated?

A static method can answer the first question, but it does not automatically know the answer to the second.


# Tutorial Problem 3 — Shared Configuration, Subclasses, and Shadowing

Class attributes are often used for configuration shared by every instance.

That is useful, but it introduces an important distinction:

- **reading** a class attribute through an instance,
- **assigning** an attribute on an instance.

We will explore that distinction step by step.


## Scenario

Imagine a query builder with a default row limit.

Every new query should use the same default unless a subclass defines a different policy.


In [13]:
class Query:
    default_limit = 100

    def __init__(self, table):
        self.table = table

    def sql(self):
        return f"SELECT * FROM {self.table} LIMIT {self.default_limit}"


q1 = Query("users")
q2 = Query("orders")

print(q1.sql())
print(q2.sql())


SELECT * FROM users LIMIT 100
SELECT * FROM orders LIMIT 100


The instances do not contain `default_limit` yet.

Python finds it on the class.


In [14]:
print("q1.__dict__:", q1.__dict__)
print("q2.__dict__:", q2.__dict__)
print("Query.default_limit:", Query.default_limit)


q1.__dict__: {'table': 'users'}
q2.__dict__: {'table': 'orders'}
Query.default_limit: 100


## Step 1 — Add class-level configuration behavior

Changing shared configuration should be a class-level operation.

That means a class method is a natural fit.


In [15]:
class Query:
    default_limit = 100

    def __init__(self, table):
        self.table = table

    @classmethod
    def set_default_limit(cls, value):
        value = int(value)
        if value <= 0:
            raise ValueError("default limit must be positive")
        cls.default_limit = value

    def sql(self):
        return f"SELECT * FROM {self.table} LIMIT {self.default_limit}"


q1 = Query("users")
q2 = Query("orders")

Query.set_default_limit(250)

print(q1.sql())
print(q2.sql())


SELECT * FROM users LIMIT 250
SELECT * FROM orders LIMIT 250


Because neither instance has its own `default_limit`, both immediately observe the changed class value.


## Step 2 — Let a subclass have a different policy

A class method receives whichever class is used for the call.

That means a subclass can acquire its own class attribute without changing the base class.


In [16]:
class ReportingQuery(Query):
    pass


ReportingQuery.set_default_limit(5000)

print("Query.default_limit:", Query.default_limit)
print("ReportingQuery.default_limit:", ReportingQuery.default_limit)

regular = Query("users")
report = ReportingQuery("events")

print(regular.sql())
print(report.sql())


Query.default_limit: 250
ReportingQuery.default_limit: 5000
SELECT * FROM users LIMIT 250
SELECT * FROM events LIMIT 5000


## Step 3 — The shadowing trap

Now assign through one instance:


In [17]:
regular.default_limit = 7

print("regular.default_limit:", regular.default_limit)
print("Query.default_limit:", Query.default_limit)
print("regular.__dict__:", regular.__dict__)


regular.default_limit: 7
Query.default_limit: 250
regular.__dict__: {'table': 'users', 'default_limit': 7}


The assignment did **not** change the class.

It created an instance attribute named `default_limit`.

That new value now shadows the class attribute during lookup.


## Step 4 — Why a class method should assign through `cls`

Compare these two ideas:

```python
self.default_limit = value
```

and:

```python
cls.default_limit = value
```

The first creates or updates object state.

The second updates the class receiving the class-method call.

When the intent is shared configuration, `cls` expresses the design correctly.


## Problem 3 challenge

Add `reset_default_limit()` so each class can return to a standard value of `100`.

Then make a subclass whose standard value is `1000`.

The reset method should respect the subclass's own standard.


### Solution


In [18]:
class Query:
    STANDARD_LIMIT = 100
    default_limit = STANDARD_LIMIT

    def __init__(self, table):
        self.table = table

    @classmethod
    def set_default_limit(cls, value):
        value = int(value)
        if value <= 0:
            raise ValueError("default limit must be positive")
        cls.default_limit = value

    @classmethod
    def reset_default_limit(cls):
        cls.default_limit = cls.STANDARD_LIMIT

    def sql(self):
        return f"SELECT * FROM {self.table} LIMIT {self.default_limit}"


class ReportingQuery(Query):
    STANDARD_LIMIT = 1000
    default_limit = STANDARD_LIMIT


Query.set_default_limit(25)
ReportingQuery.set_default_limit(9000)

Query.reset_default_limit()
ReportingQuery.reset_default_limit()

print(Query.default_limit)
print(ReportingQuery.default_limit)

assert Query.default_limit == 100
assert ReportingQuery.default_limit == 1000


100
1000


# Tutorial Problem 4 — When a Static Method Is the Right Tool

Class methods are useful, but not every operation inside a class needs access to `cls`.

Suppose a class accepts product identifiers in forms such as:

```text
"  ab-120  "
"ZX_9"
```

Normalization is conceptually related to the class, but the transformation itself needs no instance and no class state.


## Step 1 — Write the transformation as ordinary logic

First, ignore decorators.


In [19]:
def normalize_product_code(value):
    if not isinstance(value, str):
        raise TypeError("product code must be a string")

    value = value.strip().upper()
    value = value.replace("_", "-")
    return value


print(normalize_product_code("  ab_120  "))


AB-120


## Step 2 — Decide ownership

This function could remain at module level.

But imagine the transformation is only meaningful as part of the `ProductCode` API.  
Keeping it on the class can improve discoverability:

```python
ProductCode.normalize(...)
```

Because the logic does not need `self` or `cls`, `staticmethod` expresses that dependency accurately.


In [20]:
class ProductCode:
    def __init__(self, value):
        value = self.normalize(value)
        if not self.is_valid(value):
            raise ValueError(f"invalid product code: {value!r}")
        self.value = value

    @staticmethod
    def normalize(value):
        if not isinstance(value, str):
            raise TypeError("product code must be a string")

        return value.strip().upper().replace("_", "-")

    @staticmethod
    def is_valid(value):
        parts = value.split("-")
        return (
            len(parts) == 2
            and parts[0].isalpha()
            and parts[1].isdigit()
        )

    def __repr__(self):
        return f"ProductCode({self.value!r})"


code1 = ProductCode("  ab_120 ")
print(code1)


ProductCode('AB-120')


## Step 3 — Now introduce subclass configuration

Suppose different product families permit different prefixes.

The validation rule now depends on class-level configuration.

This changes the correct method type.


In [21]:
class ProductCode:
    allowed_prefixes = {"AB", "ZX"}

    def __init__(self, value):
        value = self.normalize(value)
        self.validate(value)
        self.value = value

    @staticmethod
    def normalize(value):
        if not isinstance(value, str):
            raise TypeError("product code must be a string")
        return value.strip().upper().replace("_", "-")

    @classmethod
    def validate(cls, value):
        parts = value.split("-")

        if len(parts) != 2:
            raise ValueError("code must have PREFIX-NUMBER form")

        prefix, number = parts

        if prefix not in cls.allowed_prefixes:
            raise ValueError(
                f"prefix {prefix!r} is not valid for {cls.__name__}"
            )

        if not number.isdigit():
            raise ValueError("number part must contain digits only")


class InternalProductCode(ProductCode):
    allowed_prefixes = {"INT", "DEV"}


public = ProductCode("ab-101")
internal = InternalProductCode("dev-7")

print(public.value)
print(internal.value)


AB-101
DEV-7


## Why did `validate` change?

Originally, validation depended only on its arguments.

After introducing subclass-overridable configuration, validation needed the runtime class.

That is the signal to move from `staticmethod` to `classmethod`.

A method's decorator should reflect its **current dependency**, not what it used to do.


# Tutorial Problem 5 — Build a Class-Based Parser Registry

This problem combines several ideas.

We want an API like:

```python
Parser.create("csv")
Parser.create("pipe")
```

The base class should choose a registered parser subclass.

We will build the design in stages.


## Step 1 — Start with an explicit conditional

A first attempt might look like this:


In [22]:
class CsvParser:
    def parse(self, text):
        return [piece.strip() for piece in text.split(",")]


class PipeParser:
    def parse(self, text):
        return [piece.strip() for piece in text.split("|")]


def create_parser_v1(name):
    name = name.strip().lower()

    if name == "csv":
        return CsvParser()
    elif name == "pipe":
        return PipeParser()
    else:
        raise ValueError(f"unknown parser: {name!r}")


print(create_parser_v1("csv").parse("a, b, c"))


['a', 'b', 'c']


This works, but every new parser requires editing the factory.

A registry lets classes register themselves or be registered externally.


## Step 2 — Decide what belongs on the base class

We need:

- a registry shared at class level,
- a method to register parser classes,
- a factory to construct the chosen parser,
- a small normalization helper for names.

Which method types fit those responsibilities?

- registry mutation → class method
- factory → class method
- name normalization → static method


In [23]:
class Parser:
    _registry = {}

    @staticmethod
    def normalize_name(name):
        if not isinstance(name, str):
            raise TypeError("parser name must be a string")

        normalized = name.strip().lower()

        if not normalized:
            raise ValueError("parser name cannot be empty")

        return normalized

    @classmethod
    def register(cls, name, parser_cls):
        key = cls.normalize_name(name)

        if key in cls._registry:
            raise ValueError(f"parser {key!r} is already registered")

        if not issubclass(parser_cls, Parser):
            raise TypeError("parser_cls must inherit from Parser")

        cls._registry[key] = parser_cls

    @classmethod
    def create(cls, name, **kwargs):
        key = cls.normalize_name(name)

        try:
            parser_cls = cls._registry[key]
        except KeyError:
            raise ValueError(f"unknown parser: {name!r}") from None

        return parser_cls(**kwargs)

    def parse(self, text):
        raise NotImplementedError


## Step 3 — Add concrete parser classes


In [24]:
class CsvParser(Parser):
    def __init__(self, delimiter=","):
        self.delimiter = delimiter

    def parse(self, text):
        return [
            piece.strip()
            for piece in text.split(self.delimiter)
        ]


class PipeParser(Parser):
    def parse(self, text):
        return [
            piece.strip()
            for piece in text.split("|")
        ]


## Step 4 — Register and construct


In [25]:
Parser.register("csv", CsvParser)
Parser.register("pipe", PipeParser)

p1 = Parser.create(" CSV ")
p2 = Parser.create("pipe")

print(type(p1).__name__, p1.parse("a,b,c"))
print(type(p2).__name__, p2.parse("x|y|z"))


CsvParser ['a', 'b', 'c']
PipeParser ['x', 'y', 'z']


## Step 5 — Let a subclass maintain an independent registry

The implementation above stores `_registry` on the base class.

That can be intentional, but sometimes a subclass should own a separate registry.

We can create one simply by overriding the class attribute.


In [26]:
class StrictParser(Parser):
    _registry = {}


class SemicolonParser(StrictParser):
    def parse(self, text):
        return [piece.strip() for piece in text.split(";")]


StrictParser.register("semicolon", SemicolonParser)

print("Parser registry:", sorted(Parser._registry))
print("StrictParser registry:", sorted(StrictParser._registry))

strict = StrictParser.create("semicolon")
print(strict.parse("one; two; three"))


Parser registry: ['csv', 'pipe']
StrictParser registry: ['semicolon']
['one', 'two', 'three']


## Problem 5 takeaway

Class methods are especially useful for APIs where behavior belongs to a **family of classes**, not to one particular object.

Because they receive `cls`, they can naturally work with:

- subclass-specific registries,
- subclass-specific configuration,
- subclass-aware construction.


# Tutorial Problem 6 — Replace Manual Registration with a Class Decorator

The previous registry still requires calls such as:

```python
Parser.register("csv", CsvParser)
```

Let's design a small class decorator that registers a parser.

This problem is useful because it forces us to separate two responsibilities:

- normalization/validation of the registration name,
- mutation of the class registry.


## Step 1 — Add a decorator-producing class method

The method will be called like this:

```python
@AutoParser.register_as("words")
class WordParser(AutoParser):
    ...
```

A class method is appropriate because the decorator should register the subclass into the registry of the class on which `register_as` was invoked.


In [27]:
class AutoParser:
    _registry = {}

    @staticmethod
    def normalize_name(name):
        if not isinstance(name, str):
            raise TypeError("name must be a string")

        name = name.strip().lower()

        if not name:
            raise ValueError("name cannot be empty")

        return name

    @classmethod
    def register_as(cls, name):
        key = cls.normalize_name(name)

        def decorator(parser_cls):
            if not issubclass(parser_cls, AutoParser):
                raise TypeError("registered class must inherit AutoParser")

            if key in cls._registry:
                raise ValueError(f"duplicate parser name: {key!r}")

            cls._registry[key] = parser_cls
            return parser_cls

        return decorator

    @classmethod
    def create(cls, name):
        key = cls.normalize_name(name)

        try:
            parser_cls = cls._registry[key]
        except KeyError:
            raise ValueError(f"unknown parser: {name!r}") from None

        return parser_cls()

    def parse(self, text):
        raise NotImplementedError


## Step 2 — Use the decorator


In [28]:
@AutoParser.register_as("words")
class WordParser(AutoParser):
    def parse(self, text):
        return text.split()


@AutoParser.register_as("lines")
class LineParser(AutoParser):
    def parse(self, text):
        return text.splitlines()


word_parser = AutoParser.create("words")
line_parser = AutoParser.create("lines")

print(word_parser.parse("class static instance"))
print(line_parser.parse("one\ntwo\nthree"))


['class', 'static', 'instance']
['one', 'two', 'three']


## Step 3 — Inspect the result

The decorator returns the original class after adding it to the registry.

So the class name still refers to the class normally.


In [29]:
print(WordParser)
print(AutoParser._registry["words"] is WordParser)

assert AutoParser._registry["words"] is WordParser


<class '__main__.WordParser'>
True


## Design note

This is an advanced use of a class method: it does not have to construct an instance directly.

The important point is that the operation needs to know **which class owns the registry**.


# Tutorial Problem 7 — Track `cls` Through Inheritance and `super()`

Class methods become especially interesting when inheritance and `super()` are involved.

A common misunderstanding is that calling a parent implementation through `super()` changes `cls` to the parent.

Let's test that carefully.


## Step 1 — Base implementation


In [30]:
class Message:
    category = "generic"

    @classmethod
    def describe(cls):
        return {
            "cls": cls.__name__,
            "category": cls.category,
        }


print(Message.describe())


{'cls': 'Message', 'category': 'generic'}


## Step 2 — Override only the data, not the method


In [31]:
class ErrorMessage(Message):
    category = "error"


print(ErrorMessage.describe())


{'cls': 'ErrorMessage', 'category': 'error'}


The inherited class method binds to `ErrorMessage`, not `Message`.

So the inherited method sees subclass data through `cls`.


## Step 3 — Override the method and call `super()`

Predict the value of `cls` inside the parent implementation.


In [32]:
class DetailedErrorMessage(ErrorMessage):
    severity = "high"

    @classmethod
    def describe(cls):
        data = super().describe()
        data["severity"] = cls.severity
        return data


print(DetailedErrorMessage.describe())


{'cls': 'DetailedErrorMessage', 'category': 'error', 'severity': 'high'}


The parent implementation still receives the runtime subclass.

That is why this works:

```python
super().describe()
```

The implementation changes, but the bound class remains `DetailedErrorMessage`.


## Step 4 — Prove it with explicit logging


In [33]:
class ParentTrace:
    @classmethod
    def trace(cls):
        print("ParentTrace.trace received:", cls.__name__)
        return cls.__name__


class ChildTrace(ParentTrace):
    @classmethod
    def trace(cls):
        print("ChildTrace.trace received:", cls.__name__)
        return super().trace()


result = ChildTrace.trace()

assert result == "ChildTrace"


ChildTrace.trace received: ChildTrace
ParentTrace.trace received: ChildTrace


## Problem 7 takeaway

Class methods and `super()` work together naturally for extensible construction and configuration pipelines.

The method implementation may come from a base class while `cls` continues to represent the active subclass.


# Tutorial Problem 8 — Build an Inheritance-Aware Validation Pipeline

We will now combine:

- static helpers,
- class-level policy,
- alternative constructors,
- `super()`.

The domain will be importable records.


## Goal

We want:

```python
UserRecord.from_mapping(...)
AdminRecord.from_mapping(...)
```

Both classes share basic validation.

`AdminRecord` adds one extra rule.

The factory must return the correct subclass.


## Step 1 — Identify pure helpers

Checking whether a value is a non-empty string needs no instance and no class state.

That makes it a good static method candidate.


In [34]:
class Record:
    @staticmethod
    def require_text(value, field_name):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(
                f"{field_name} must be a non-empty string"
            )
        return value.strip()


print(Record.require_text("  Alice  ", "name"))


Alice


## Step 2 — Add class-dependent validation

Suppose each record class declares the fields it requires.

Validation now needs `cls.required_fields`.


In [35]:
class Record:
    required_fields = ()

    @staticmethod
    def require_text(value, field_name):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(
                f"{field_name} must be a non-empty string"
            )
        return value.strip()

    @classmethod
    def validate_mapping(cls, mapping):
        if not isinstance(mapping, dict):
            raise TypeError("mapping must be a dict")

        cleaned = {}

        for field in cls.required_fields:
            cleaned[field] = cls.require_text(
                mapping.get(field),
                field
            )

        return cleaned


## Step 3 — Add subclass-aware construction

The constructor method should call `cls(...)`, not `Record(...)`.


In [36]:
class UserRecord(Record):
    required_fields = ("username", "email")

    def __init__(self, username, email):
        self.username = username
        self.email = email

    @classmethod
    def from_mapping(cls, mapping):
        cleaned = cls.validate_mapping(mapping)
        return cls(**cleaned)


user = UserRecord.from_mapping({
    "username": "  alice ",
    "email": " alice@example.com ",
})

print(user.__dict__)


{'username': 'alice', 'email': 'alice@example.com'}


## Step 4 — Let a subclass extend validation

An administrator needs a role.

We can override class data and extend the class-method pipeline.


In [37]:
class AdminRecord(UserRecord):
    required_fields = ("username", "email", "role")

    @classmethod
    def validate_mapping(cls, mapping):
        cleaned = super().validate_mapping(mapping)

        allowed_roles = {"reader", "editor", "owner"}

        role = cleaned["role"].lower()

        if role not in allowed_roles:
            raise ValueError(
                f"role must be one of {sorted(allowed_roles)}"
            )

        cleaned["role"] = role
        return cleaned

    def __init__(self, username, email, role):
        super().__init__(username, email)
        self.role = role


admin = AdminRecord.from_mapping({
    "username": "bob",
    "email": "bob@example.com",
    "role": " OWNER ",
})

print(type(admin).__name__)
print(admin.__dict__)

assert type(admin) is AdminRecord
assert admin.role == "owner"


AdminRecord
{'username': 'bob', 'email': 'bob@example.com', 'role': 'owner'}


## Step 5 — Why did the inherited constructor still work?

`AdminRecord` inherits `from_mapping`.

When called as:

```python
AdminRecord.from_mapping(...)
```

the class method binds `cls` to `AdminRecord`.

Therefore:

```python
cls.validate_mapping(...)
```

uses the subclass validation, and:

```python
cls(**cleaned)
```

constructs the subclass.

This is polymorphism at the class-method level.


# Tutorial Problem 9 — Evolve a Timer from Simple to Testable

The timer example is a particularly good place to decide which behavior belongs at each level.

We will build a new timer design gradually rather than jumping to the final class.


## Step 1 — What belongs to the instance?

Each timer needs its own:

- start time,
- stop time.

Those values cannot be class attributes because several timers may run independently.


In [38]:
from datetime import datetime, timezone, timedelta


class SimpleTimer:
    def __init__(self):
        self._start = None
        self._end = None

    def start(self):
        self._start = datetime.now(timezone.utc)
        self._end = None

    def stop(self):
        self._end = datetime.now(timezone.utc)


a = SimpleTimer()
b = SimpleTimer()

print(a.__dict__)
print(b.__dict__)


{'_start': None, '_end': None}
{'_start': None, '_end': None}


## Step 2 — What can be shared?

Suppose every timer in one class should display times in the same timezone.

That policy belongs naturally at class level.


In [39]:
class TimezoneTimer:
    display_tz = timezone.utc

    def __init__(self):
        self._start = None
        self._end = None

    @classmethod
    def set_display_timezone(cls, offset_hours, name):
        cls.display_tz = timezone(
            timedelta(hours=offset_hours),
            name
        )


TimezoneTimer.set_display_timezone(2, "UTC+2")
print(TimezoneTimer.display_tz)


UTC+2


## Step 3 — What behavior needs neither instance nor class state?

Obtaining the current UTC time does not require timer state.

We can represent the default clock as a static method.


In [40]:
class TimezoneTimer:
    display_tz = timezone.utc

    def __init__(self):
        self._start = None
        self._end = None

    @staticmethod
    def utc_now():
        return datetime.now(timezone.utc)

    @classmethod
    def set_display_timezone(cls, offset_hours, name):
        cls.display_tz = timezone(
            timedelta(hours=offset_hours),
            name
        )

    def start(self):
        self._start = self.utc_now()
        self._end = None

    def stop(self):
        self._end = self.utc_now()


This is already a reasonable design, but there is a testing problem.

A test that depends on the real clock is nondeterministic.

Using `sleep()` makes tests slow and imprecise.

We can improve the design by making the clock injectable.


## Step 4 — Inject the clock

The static method can remain the default, while each instance may receive a replacement callable.


In [41]:
class TimerError(RuntimeError):
    pass


class TestableTimer:
    display_tz = timezone.utc

    def __init__(self, *, clock=None):
        self._clock = clock or self.utc_now
        self._start = None
        self._end = None

    @staticmethod
    def utc_now():
        return datetime.now(timezone.utc)

    @classmethod
    def set_display_timezone(cls, offset_hours, name):
        cls.display_tz = timezone(
            timedelta(hours=offset_hours),
            name
        )

    def start(self):
        self._start = self._as_utc(self._clock())
        self._end = None
        return self

    def stop(self):
        if self._start is None:
            raise TimerError("timer must be started first")

        self._end = self._as_utc(self._clock())
        return self

    @staticmethod
    def _as_utc(value):
        if not isinstance(value, datetime):
            raise TypeError("clock must return a datetime")

        if value.tzinfo is None:
            raise ValueError(
                "clock must return a timezone-aware datetime"
            )

        return value.astimezone(timezone.utc)


Notice the method choices:

- `start` and `stop` mutate one timer → instance methods
- `set_display_timezone` mutates class policy → class method
- `utc_now` needs neither object nor class → static method
- `_as_utc` validates/transforms an argument only → static method


## Step 5 — Add display properties

Stored timestamps remain UTC.

Display conversion uses the class's current display policy.


In [42]:
class TestableTimer(TestableTimer):
    @property
    def start_time(self):
        if self._start is None:
            raise TimerError("timer has not been started")

        return self._start.astimezone(type(self).display_tz)

    @property
    def end_time(self):
        if self._end is None:
            raise TimerError("timer has not been stopped")

        return self._end.astimezone(type(self).display_tz)

    @property
    def elapsed(self):
        if self._start is None:
            raise TimerError("timer has not been started")

        end = self._end

        if end is None:
            end = self._as_utc(self._clock())

        return end - self._start


We used:

```python
type(self).display_tz
```

rather than hard-coding `TestableTimer.display_tz`.

That means a subclass can provide its own display policy.


## Step 6 — Build a deterministic fake clock

A fake clock will return known values in sequence.


In [43]:
class SequenceClock:
    def __init__(self, *values):
        self._values = iter(values)

    def __call__(self):
        return next(self._values)


t0 = datetime(2035, 5, 1, 12, 0, tzinfo=timezone.utc)
t1 = t0 + timedelta(seconds=3.75)

clock = SequenceClock(t0, t1)

timer = TestableTimer(clock=clock)
timer.start()
timer.stop()

print(timer.elapsed)
assert timer.elapsed == timedelta(seconds=3.75)


0:00:03.750000


No sleeping was necessary.

The test is exact because time is now a dependency we control.


## Step 7 — Add a subclass-aware convenience constructor

We want:

```python
timer = TestableTimer.started(clock=...)
```

and we want the same method to work correctly for subclasses.

That is another class-method use case.


In [44]:
class TestableTimer(TestableTimer):
    @classmethod
    def started(cls, *, clock=None):
        return cls(clock=clock).start()


## Step 8 — Prove that `started` respects subclasses


In [45]:
class EastTimer(TestableTimer):
    pass


EastTimer.set_display_timezone(4, "UTC+4")

clock = SequenceClock(
    t0,
    t0 + timedelta(seconds=2),
)

east = EastTimer.started(clock=clock)
east.stop()

print(type(east).__name__)
print(east.start_time)
print(east.elapsed)

assert type(east) is EastTimer
assert east.start_time.utcoffset() == timedelta(hours=4)


EastTimer
2035-05-01 16:00:00+04:00
0:00:02


## Problem 9 conclusion

The final design did not choose decorators by appearance.

Each choice followed the location of the required state:

- timer-specific state → instance
- class-wide policy / subclass-aware construction → class
- argument-only helper logic → static


# Tutorial Problem 10 — Reconstruct `staticmethod` and `classmethod` Behavior Conceptually

This is a reasoning exercise.

We will not reimplement Python's built-ins completely, but we will build small descriptors that mimic the most important binding behavior.

This connects the earlier experiments to the descriptor protocol.


## Step 1 — A simplified static-method descriptor

Its job is simple:

> when accessed, return the stored function without binding an instance or class.


In [46]:
class MiniStaticMethod:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        return self.func


class StaticExperiment:
    def utility(x):
        return x * 10

    utility = MiniStaticMethod(utility)


obj = StaticExperiment()

print(StaticExperiment.utility(3))
print(obj.utility(3))

assert StaticExperiment.utility is obj.utility


30
30


## Step 2 — A simplified class-method descriptor

A class method must return a callable that inserts the owner class as the first argument.


In [47]:
from types import MethodType


class MiniClassMethod:
    def __init__(self, func):
        self.func = func

    def __get__(self, instance, owner):
        return MethodType(self.func, owner)


class ClassExperiment:
    label = "base"

    def identify(cls):
        return cls.__name__, cls.label

    identify = MiniClassMethod(identify)


class ChildClassExperiment(ClassExperiment):
    label = "child"


print(ClassExperiment.identify())
print(ChildClassExperiment.identify())

assert ChildClassExperiment.identify() == (
    "ChildClassExperiment",
    "child"
)


('ClassExperiment', 'base')
('ChildClassExperiment', 'child')


## What this teaches

The important behavior is implemented during **attribute access**.

That is why merely retrieving:

```python
obj.method
```

can produce a bound object before the method is ever called.


# Tutorial Problem 11 — Method-Type Code Review

Consider this intentionally questionable design:


In [48]:
class InvoiceBad:
    tax_rate = 0.20

    def __init__(self, amounts):
        self.amounts = list(amounts)

    @staticmethod
    def subtotal(invoice):
        return sum(invoice.amounts)

    @staticmethod
    def tax(amount):
        return amount * InvoiceBad.tax_rate

    @staticmethod
    def from_strings(values):
        return InvoiceBad([float(v) for v in values])


It runs, but method type is being used poorly.

Let's review one method at a time.


## Review A — `subtotal`

`subtotal` needs one particular invoice's `amounts`.

Passing the object manually is unnecessary:

```python
InvoiceBad.subtotal(invoice)
```

This is exactly what an instance method already models.

So `subtotal` should be an instance method.


## Review B — `tax`

`tax` uses class policy: `tax_rate`.

If subclasses may have different rates, a class method is a better fit than a static method with a hard-coded class reference.


## Review C — `from_strings`

This is an alternative constructor.

Hard-coding `InvoiceBad(...)` prevents subclass-aware construction.

It should usually be a class method returning `cls(...)`.


## Refactored solution


In [49]:
class Invoice:
    tax_rate = 0.20

    def __init__(self, amounts):
        self.amounts = [float(x) for x in amounts]

    def subtotal(self):
        return sum(self.amounts)

    @classmethod
    def tax_for(cls, amount):
        return float(amount) * cls.tax_rate

    @classmethod
    def from_strings(cls, values):
        return cls([float(v) for v in values])

    def total(self):
        subtotal = self.subtotal()
        return subtotal + type(self).tax_for(subtotal)


class ReducedTaxInvoice(Invoice):
    tax_rate = 0.05


regular = Invoice.from_strings(["10", "20"])
reduced = ReducedTaxInvoice.from_strings(["10", "20"])

print(type(regular).__name__, regular.total())
print(type(reduced).__name__, reduced.total())

assert regular.total() == 36.0
assert reduced.total() == 31.5


Invoice 36.0
ReducedTaxInvoice 31.5


## Code-review rule

A method can be *technically callable* while still communicating the wrong ownership.

Good method design makes dependencies visible:

- `self` tells readers the behavior uses object state,
- `cls` tells readers it uses class/subclass state,
- neither tells readers the logic is independent of both.


# Tutorial Problem 12 — Capstone: Extensible Event Decoder

We will finish with a larger problem that combines the full topic.

An application receives event dictionaries:

```python
{
    "type": "login",
    "payload": {...}
}
```

Different event types should produce different event subclasses.

We want this API:

```python
Event.decode(data)
```

The system should:

- normalize event type names,
- register event subclasses,
- create the correct subclass,
- allow subclass-specific payload validation,
- preserve clean separation between instance, class, and static behavior.


## Step 1 — Decide where each piece of state lives

We need a registry of event types.

That is shared by the event class family, so it belongs at class level.

Each decoded event also has its own payload.

That belongs on the instance.


## Step 2 — Build the base event


In [50]:
class Event:
    _registry = {}

    def __init__(self, payload):
        self.payload = payload

    @staticmethod
    def normalize_type_name(name):
        if not isinstance(name, str):
            raise TypeError("event type must be a string")

        normalized = name.strip().lower().replace("-", "_")

        if not normalized:
            raise ValueError("event type cannot be empty")

        return normalized


The normalization helper is static because it depends only on its argument.


## Step 3 — Add registration

Registration mutates shared class state, so we use a class method.


In [51]:
class Event(Event):
    @classmethod
    def register(cls, type_name, event_cls):
        key = cls.normalize_type_name(type_name)

        if not issubclass(event_cls, Event):
            raise TypeError("event_cls must inherit Event")

        if key in cls._registry:
            raise ValueError(
                f"event type {key!r} is already registered"
            )

        cls._registry[key] = event_cls


## Step 4 — Add a validation hook

Every event subclass may validate its payload differently.

The base implementation only checks that the payload is a dictionary.

The method is a class method because a subclass may override class-level policy or behavior.


In [52]:
class Event(Event):
    @classmethod
    def validate_payload(cls, payload):
        if not isinstance(payload, dict):
            raise TypeError("payload must be a dict")

        return dict(payload)


## Step 5 — Add the decoder factory

The factory selects a class from the registry, asks that class to validate its payload, then constructs it.


In [53]:
class Event(Event):
    @classmethod
    def decode(cls, data):
        if not isinstance(data, dict):
            raise TypeError("event data must be a dict")

        type_name = cls.normalize_type_name(data.get("type", ""))

        try:
            event_cls = cls._registry[type_name]
        except KeyError:
            raise ValueError(
                f"unknown event type: {type_name!r}"
            ) from None

        payload = event_cls.validate_payload(
            data.get("payload", {})
        )

        return event_cls(payload)


## Step 6 — Add concrete event subclasses


In [54]:
class LoginEvent(Event):
    @classmethod
    def validate_payload(cls, payload):
        payload = super().validate_payload(payload)

        username = payload.get("username")

        if not isinstance(username, str) or not username.strip():
            raise ValueError("login event requires username")

        payload["username"] = username.strip()
        return payload


class PurchaseEvent(Event):
    @classmethod
    def validate_payload(cls, payload):
        payload = super().validate_payload(payload)

        amount = float(payload.get("amount", 0))

        if amount <= 0:
            raise ValueError(
                "purchase amount must be greater than zero"
            )

        payload["amount"] = amount
        return payload


## Step 7 — Register them


In [55]:
Event.register("login", LoginEvent)
Event.register("purchase", PurchaseEvent)

print(Event._registry)


{'login': <class '__main__.LoginEvent'>, 'purchase': <class '__main__.PurchaseEvent'>}


## Step 8 — Decode real data


In [56]:
login = Event.decode({
    "type": " LOGIN ",
    "payload": {
        "username": "  alice  "
    }
})

purchase = Event.decode({
    "type": "purchase",
    "payload": {
        "amount": "19.99"
    }
})

print(type(login).__name__, login.payload)
print(type(purchase).__name__, purchase.payload)

assert type(login) is LoginEvent
assert login.payload["username"] == "alice"

assert type(purchase) is PurchaseEvent
assert purchase.payload["amount"] == 19.99


LoginEvent {'username': 'alice'}
PurchaseEvent {'amount': 19.99}


## Step 9 — Add a subclass-specific registry

Suppose an internal subsystem should accept a different set of events.

Because registry access is class-based, a subclass can own a separate registry simply by replacing the class attribute.


In [57]:
class InternalEvent(Event):
    _registry = {}


class HealthCheckEvent(InternalEvent):
    pass


InternalEvent.register("health", HealthCheckEvent)

internal = InternalEvent.decode({
    "type": "health",
    "payload": {"status": "ok"}
})

print(type(internal).__name__)
print("public registry:", sorted(Event._registry))
print("internal registry:", sorted(InternalEvent._registry))

assert type(internal) is HealthCheckEvent
assert "health" not in Event._registry


HealthCheckEvent
public registry: ['login', 'purchase']
internal registry: ['health']


## Step 10 — Method-type audit

Let's classify the final design.

### Instance behavior

```python
__init__
```

The payload belongs to one event object.

### Class behavior

```python
register
validate_payload
decode
```

These operations depend on the active class hierarchy, registries, subclass validation, or subclass construction.

### Static behavior

```python
normalize_type_name
```

This transformation needs only its argument.

That division is the main design skill this notebook has been practicing.


# Extra Guided Exercise A — Predict Before Running

For each expression, decide what object is automatically supplied.

```python
obj.run()
Type.run(obj)
Type.build()
obj.build()
Type.clean(value)
obj.clean(value)
```

Assume:

- `run` is a normal instance method,
- `build` is a class method,
- `clean` is a static method.


## Solution

- `obj.run()` → Python binds `obj` as the first argument.
- `Type.run(obj)` → no automatic instance is supplied by class lookup; you supply `obj` explicitly.
- `Type.build()` → Python supplies `Type` as `cls`.
- `obj.build()` → Python still supplies the class, not `obj`, as `cls`.
- `Type.clean(value)` → nothing is injected.
- `obj.clean(value)` → nothing is injected.


# Extra Guided Exercise B — Find the Hidden Inheritance Bug

What is wrong with this constructor?


In [58]:
class ShapeBad:
    @classmethod
    def from_size(cls, size):
        return ShapeBad(size)

    def __init__(self, size):
        self.size = size


class CircleBad(ShapeBad):
    pass


bad_circle = CircleBad.from_size(5)
print(type(bad_circle).__name__)


ShapeBad


## Solution

The method receives `cls` but ignores it.

That defeats the main benefit of using a class method.

The corrected version should construct with:

```python
return cls(size)
```


In [59]:
class Shape:
    def __init__(self, size):
        self.size = size

    @classmethod
    def from_size(cls, size):
        return cls(size)


class Circle(Shape):
    pass


circle = Circle.from_size(5)

print(type(circle).__name__)
assert type(circle) is Circle


Circle


# Extra Guided Exercise C — Static Method or Module Function?

Suppose this helper appears inside `Customer`:

```python
@staticmethod
def sha256_bytes(data):
    ...
```

Is `staticmethod` wrong?

Not necessarily.

The more useful question is:

> Is hashing bytes part of the cohesive responsibility of `Customer`?

If many unrelated classes need the same helper, a module-level utility is probably cleaner.

`staticmethod` solves a binding question.  
It does **not** automatically solve an API-ownership question.


# Final Decision Checklist

When you write a method inside a class, ask these questions in order.

### 1. Does the operation require one object's state?

Examples:

- reading `self.items`
- updating `self.status`
- calculating a result from instance fields

Use an **instance method**.

### 2. Does the operation need the runtime class or subclass?

Examples:

- constructing `cls(...)`
- reading `cls.config`
- changing class-wide policy
- accessing a class registry
- allowing subclasses to customize behavior

Use a **class method**.

### 3. Does the operation need neither?

Examples:

- formatting an argument
- checking argument syntax
- converting one representation to another

Consider a **static method**.

### 4. Is the helper really part of the class's responsibility?

If not, prefer a **module-level function** or a separate collaborator.

This final question is about software design, not binding mechanics.


# Final Practice Project — Do Not Read a Solution First

Create a document-import system with:

```python
Document.from_text(...)
Document.from_mapping(...)
Document.decode(...)
```

Requirements:

- each document instance stores its own content,
- subclasses declare a document type,
- a registry maps type names to subclasses,
- type-name normalization is a pure helper,
- each subclass may define its own required fields,
- `decode(...)` must return the correct subclass,
- a subclass should be able to maintain a separate registry,
- add at least three assertions proving correct class binding,
- inspect at least one method's `__self__`,
- inspect at least one raw descriptor through `__dict__`.

Suggested subclasses:

- `NoteDocument`
- `ArticleDocument`
- `LogDocument`

Before implementing a method, write one sentence explaining **why** it should be an instance, class, or static method.
